# IC-0921 — How diversified is your client, really?

**ML & FinTech · 115-1 · 20260921 · 50 minutes: 40 working, 10 discussion**

| Step | Minutes | What you do |
|---|---|---|
| 1 | 12 | Learn the tools on eight customers |
| 2 | 16 | Cluster a real client's portfolio, two ways |
| 3 | 12 | Give the client advice |
| — | 10 | Discussion |

You work at a robo-advisor. A client calls:

> *"I'm well diversified. I own twelve different stocks."*

Twelve tickers is not twelve bets. If several of them rise and fall together, the client owns
fewer independent positions than he thinks. **By the end of this exercise you will tell him how
many bets he actually owns.**

You may use any AI tool. It will write the code for you. The code is not what is graded — whether
you understand what the output means is.

## Setup

Run this first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

---
# Step 1 — Learn the tools (12 min)

Before the real portfolio, practise on something small where you can see the answer with your
own eyes: **eight credit card customers** of a digital bank, described by how much they spend
and how often they use the card.

The code below is complete. **Run each cell, read the output, and answer the questions.**
You will reuse exactly these functions in Step 2.

In [ ]:
# Eight credit card customers of a small digital bank
cust = pd.DataFrame({
    "customer": ["Amy", "Ben", "Cora", "Dan", "Eve", "Fay", "Gus", "Hal"],
    "spend":    [5, 6, 4, 5, 28, 30, 32, 29],   # monthly card spending, NT$ thousand
    "trans":    [4, 5, 3, 6, 22, 25, 20, 24],   # card transactions per month
}).set_index("customer")
cust

### 1a. k-means

`fit_predict` gives each customer a cluster number.

In [ ]:
# 1a. k-means with K = 2
labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(cust)
cust["kmeans"] = labels
cust

### 1b. Look at the result

In [ ]:
# 1b. plot the customers, coloured by their cluster
plt.scatter(cust["spend"], cust["trans"], c=cust["kmeans"], s=100)
for name, row in cust.iterrows():
    plt.annotate(name, (row["spend"], row["trans"]))
plt.xlabel("monthly spending (NT$ thousand)")
plt.ylabel("transactions per month")
plt.show()

### 1c. A dendrogram

A dendrogram shows the order in which customers join into groups. Read it from the bottom up.
**The height at which two branches join tells you how different they are.**

In [ ]:
# 1c. hierarchical clustering and its dendrogram
Z = linkage(cust[["spend", "trans"]].values, method="average")
dendrogram(Z, labels=cust.index.tolist())
plt.ylabel("height at which groups join")
plt.show()

### 1d. Cut the tree into groups

In [ ]:
# 1d. cut the tree into a chosen number of groups
cust["tree"] = fcluster(Z, 2, criterion="maxclust")
cust

### Step 1 questions

**Q1.** k-means gave the light spenders one number and the heavy spenders the other. Does the
number itself (0 or 1) mean anything? Why or why not?

> _replace this line_

**Q2.** In the dendrogram, the two final groups join very high up, while everything before that
joined low down. What does that big jump tell you about how many groups these customers really
form?

> _replace this line_

**Q3.** Change `n_clusters=2` to `n_clusters=3` in 1a and run it again. Who moves into the new
cluster? Looking at the plot, is that new cluster a real group or is k-means forcing it?

> _replace this line_

In [ ]:
# Q3 — try K = 3 here



---
# Step 2 — Your client's portfolio (16 min)

Now the real data: every trading day of 2017 for the client's twelve holdings.

In [ ]:
from pathlib import Path
CAND = ["historical_stock_prices_2017.csv",
        "../slides/historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), None)
if path is None:
    raise FileNotFoundError("Put historical_stock_prices_2017.csv next to this notebook.")

prices = pd.read_csv(path, index_col=0).dropna(axis=1)

# Your client's 12 holdings
PORTFOLIO = ["BANC","BANF","GWB","ISTR","CAKE","DFRG","LOCO","BREW",
             "AAPL","EBAY","QQQ","CCOI"]

P = prices[PORTFOLIO]        # 251 trading days x 12 stocks, 2017
print(P.shape)
P.head(3)

**One thing to set up first.** You are clustering **stocks**, not days. Each stock must be
one row, like each customer was one row in Step 1. `P` has the stocks in its columns, so use
`P.T` (the transpose) whenever you cluster the stocks.

---
## 2a. The obvious approach (6 min)

Run **k-means with K = 3** on the stock price series, exactly as they are in the file.
Print which stocks land in each cluster.

In [ ]:
# 2a — reuse the k-means line from Step 1, on P.T



**Q4.** Look at the stocks inside each of your three clusters.
What do they actually have in common? Be specific. If you cannot find anything they share as
*businesses*, say so, and say what they share instead.

*Hint: look at `P.mean()` next to your clusters.*

> _replace this line_

**Q5.** Would you show these three groups to the client as his three bets? Yes / No, and why.

> _replace this line_

---
## 2b. Cluster on what actually matters (10 min)

Two stocks belong in the same bet when they **move together**, not when they cost the same.

```python
R = np.log(P).diff().dropna()      # daily log returns
C = R.corr()                       # how much each pair moves together
D = 1 - C                          # turn correlation into a distance
Z = linkage(squareform(D.values, checks=False), method="average")
dendrogram(Z, labels=D.columns.tolist())
```

Draw the dendrogram, then cut it into 3 groups with `fcluster`, as in Step 1.

In [ ]:
# 2b



**Q6.** Write down your three groups.

> _____

**Q7.** One stock joins the others only at the very top. Which one? What does that tell you
about it?

> _____

---
# Step 3 — Advise the client (12 min)

**Q8.** How many independent bets does the client actually own?

> _____

**Q9.** Look at AAPL and QQQ in your dendrogram. Should the client hold both? Why?
*(QQQ is a fund that tracks the 100 largest Nasdaq companies.)*

> _____

**Q10.** The client wants to cut from twelve holdings down to four. **Name the four you would
keep**, and give one sentence of reasoning based on your own dendrogram.

Four tickers. "It depends" is not an answer — he is on the phone.

> Keep: _____ , _____ , _____ , _____
>
> Because: _____

---
## Before the discussion

Restart & Run All, check every blank is filled, push as `IC-0921.ipynb`.

Be ready to answer out loud: **the client owns twelve stocks. Is he diversified?**